# EasyOCR benchmark (simple)

Measures **speed**, **Type I** (false positive), and **Type II** (false negative) on random 10-frame sequences.

Images: `images/base/` (10) and `images/with_number/` (≥1). Run all cells top to bottom.

In [ ]:
import random
import re
import time
from pathlib import Path
from statistics import mean

import easyocr
import numpy as np
from PIL import Image

# --- config ---
ROOT = Path.cwd()
BASE_DIR = ROOT / "images" / "base"
NUMBER_DIR = ROOT / "images" / "with_number"
EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

N_SLOTS = 10
N_TRIALS = 50
P_HAS_NUMBER = 0.3
RANDOM_SEED = 42
WARMUP = True

IMAGE_SCALE = 4
OCR_KWARGS = {
    "allowlist": "0123456789",
    "mag_ratio": 1.5,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.7,
}
MIN_CONFIDENCE = 0.2
USE_GPU = None  # None = auto


def list_images(folder: Path) -> list[Path]:
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXT],
        key=lambda p: p.name.lower(),
    )


def load_rgb(path: Path) -> np.ndarray:
    im = Image.open(path).convert("RGB")
    if IMAGE_SCALE != 1:
        w, h = im.size
        im = im.resize((w * IMAGE_SCALE, h * IMAGE_SCALE), Image.Resampling.LANCZOS)
    return np.array(im)


def read_frame(reader, path: Path) -> tuple[str, float, bool]:
    """Returns (ocr_text, elapsed_ms, has_digit)."""
    t0 = time.perf_counter()
    raw = reader.readtext(load_rgb(path), detail=1, paragraph=False, **OCR_KWARGS)
    ms = (time.perf_counter() - t0) * 1000.0
    parts = [
        str(x[1]).strip()
        for x in raw
        if float(x[2]) >= MIN_CONFIDENCE and str(x[1]).strip()
    ]
    text = " | ".join(parts) if parts else ""
    has_digit = any(re.search(r"\d", p) for p in parts)
    return text, ms, has_digit


def build_trial(rng, bases: list[Path], numbers: list[Path]):
    paths = list(bases)
    injected = rng.random() < P_HAS_NUMBER
    slot = None
    if injected:
        slot = rng.randrange(N_SLOTS)
        paths[slot] = rng.choice(numbers)
    return paths, injected, slot

In [ ]:
bases = list_images(BASE_DIR)
numbers = list_images(NUMBER_DIR)
if len(bases) != N_SLOTS:
    raise SystemExit(f"Need {N_SLOTS} images in {BASE_DIR}, found {len(bases)}")
if not numbers:
    raise SystemExit(f"Need at least 1 image in {NUMBER_DIR}")

gpu = USE_GPU
if gpu is None:
    try:
        import torch

        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print("Loading EasyOCR...")
t0 = time.perf_counter()
reader = easyocr.Reader(["en"], gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.1f}s (gpu={gpu})")

if WARMUP:
    read_frame(reader, bases[0])

rng = random.Random(RANDOM_SEED)
frame_ms: list[float] = []
batch_ms: list[float] = []
tp = fp = fn = tn = 0
trials_with_number = 0
trials_without_number = 0

for trial in range(N_TRIALS):
    paths, injected, inject_slot = build_trial(rng, bases, numbers)
    if injected:
        trials_with_number += 1
    else:
        trials_without_number += 1

    t_batch = time.perf_counter()
    for slot, path in enumerate(paths):
        text, ms, got_digit = read_frame(reader, path)
        frame_ms.append(ms)
        is_number_slot = injected and slot == inject_slot

        if is_number_slot and got_digit:
            tp += 1
        elif is_number_slot and not got_digit:
            fn += 1
        elif not is_number_slot and got_digit:
            fp += 1
        else:
            tn += 1
    batch_ms.append((time.perf_counter() - t_batch) * 1000.0)

n_frames = len(frame_ms)
n_number_slots = tp + fn
n_plain_slots = fp + tn
type_i_pct = 100.0 * fp / n_plain_slots if n_plain_slots else 0.0
type_ii_pct = 100.0 * fn / n_number_slots if n_number_slots else 0.0

print()
print("=" * 50)
print("EASYOCR BENCHMARK")
print("=" * 50)
print(f"trials:              {N_TRIALS}")
print(f"frames per trial:    {N_SLOTS}")
print(f"total frames:        {n_frames}")
print(f"trials w/ number:    {trials_with_number}")
print(f"trials w/o number:   {trials_without_number}")
print(f"image_scale:         {IMAGE_SCALE}")
print(f"gpu:                 {gpu}")
print()
print("SPEED")
print(f"  reader init:       {init_s:.2f} s")
print(f"  mean per frame:    {mean(frame_ms):.1f} ms")
print(f"  mean per batch:    {mean(batch_ms):.1f} ms  ({mean(batch_ms)/1000:.2f} s)")
print(f"  batch fps:         {1000.0 * N_SLOTS / mean(batch_ms):.2f}")
print()
print("ERRORS (frame-level)")
print(f"  type I  false positive:  {fp:4d}  ({type_i_pct:.1f}% of non-number frames)")
print(f"  type II false negative:    {fn:4d}  ({type_ii_pct:.1f}% of number slots)")
print(f"  true positive:             {tp:4d}")
print(f"  true negative:             {tn:4d}")
print("=" * 50)